#### EDA 07: Sellers table
**File:** `olist_sellers_dataset.csv`  |  **Goal:** understand who the sellers are and where they are, using this table only. No joins happen here. Cross-table checks are listed at the end.

**Flow:** Grain → Missing → Zip prefix → States and cities → City text quality → Consistency of zip, city and state → Summary

#### Load and first look
**Question:** how big is the table and what does a row look like?

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import unicodedata

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

RAW = Path("../../data/raw")                      # path from the notebook folder to the raw data

sellers_raw = pd.read_csv(RAW / "olist_sellers_dataset.csv")
sellers = sellers_raw.copy()                      # all changes happen on the copy, raw stays untouched

print("shape:", sellers.shape)
print(sellers.dtypes)
sellers.head(3)

shape: (3095, 4)
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


- 3,095 rows and 4 columns: seller id, zip prefix, city and state. There is no information about what the seller sells or how it performs. That comes from other tables.
- The zip prefix is stored as a number.

#### Keys, duplicates and missing values
**Question:** is `seller_id` a valid key, and which columns have gaps?

In [2]:
print("seller_id unique      :", sellers["seller_id"].is_unique)
print("fully duplicated rows :", sellers.duplicated().sum())

# Column-wise missing count and percentage, for every column (zeros included)
missing = pd.DataFrame({"n_rows": len(sellers),
                        "n_missing": sellers.isna().sum(),
                        "pct_missing": (sellers.isna().mean() * 100).round(2)})
display(missing.sort_values("n_missing", ascending=False))

seller_id unique      : True
fully duplicated rows : 0


,n_rows,n_missing,pct_missing
seller_id,3095,0,0.0
seller_zip_code_prefix,3095,0,0.0
seller_city,3095,0,0.0
seller_state,3095,0,0.0


- `seller_id` is unique and there are no duplicated rows. One row is one seller.
- No column has a missing value. Nothing to impute or flag.

#### Zip prefix
**Question:** is the zip prefix stored correctly?

In [3]:
zip_len = sellers["seller_zip_code_prefix"].astype(str).str.len().value_counts().sort_index()
print("zip prefix length (digits):")
print(zip_len.to_string())

short = sellers["seller_zip_code_prefix"] < 10000
print("\nstates of the 4-digit prefixes:", sellers.loc[short, "seller_state"].value_counts().to_dict())
print("distinct zip prefixes         :", sellers["seller_zip_code_prefix"].nunique())
print("most sellers in one prefix    :", sellers.groupby("seller_zip_code_prefix").size().max())

# Text version with the zero restored, for display and safe text joins
sellers["zip5"] = sellers["seller_zip_code_prefix"].astype(str).str.zfill(5)
sellers[["seller_zip_code_prefix", "zip5"]].head(3)

zip prefix length (digits):
seller_zip_code_prefix
4    1027
5    2068

states of the 4-digit prefixes: {'SP': 1027}
distinct zip prefixes         : 2246
most sellers in one prefix    : 49


,seller_zip_code_prefix,zip5
0,13023,13023
1,13844,13844
2,20031,20031


- The same problem as in the customers table: a number type drops the leading zero. 1,027 prefixes (33.2%) have only 4 digits, and all of them are in SP.
- 2,246 distinct prefixes serve 3,095 sellers. One prefix holds up to 49 sellers, so many sellers share a location.
- **Decision:** keep the numeric prefix for the geolocation join (that table also stores it as a number) and add `zip5` as 5-digit text for display.

#### States and cities
**Question:** where are the sellers, and how concentrated is it?

In [4]:
states = sellers["seller_state"].value_counts()
state_tbl = pd.DataFrame({"n_sellers": states,
                          "pct": (states / len(sellers) * 100).round(1),
                          "cum_pct": (states.cumsum() / len(sellers) * 100).round(1)})
print("states:", len(states))
display(state_tbl.head(6))
print("states with a single seller:", states[states == 1].index.tolist())

cities = sellers["seller_city"].value_counts()
print("\ndistinct cities               :", len(cities))
print("cities with a single seller   :", (cities == 1).sum(), f"({(cities == 1).mean() * 100:.1f}% of cities)")
cities.head(5).rename("n_sellers").to_frame()

states: 23


,n_sellers,pct,cum_pct
seller_state,,,
SP,1849,59.7,59.7
PR,349,11.3,71.0
MG,244,7.9,78.9
SC,190,6.1,85.0
RJ,171,5.5,90.6
RS,129,4.2,94.7


states with a single seller: ['AC', 'PI', 'MA', 'AM', 'PA']

distinct cities               : 611
cities with a single seller   : 342 (56.0% of cities)


,n_sellers
seller_city,
sao paulo,694
curitiba,127
rio de janeiro,96
belo horizonte,68
ribeirao preto,52


- Sellers are in 23 states, far more concentrated than customers: SP alone has 1,849 sellers (59.7%) and the top 5 states (SP, PR, MG, SC, RJ) hold 90.6%.
- 5 states (AC, PI, MA, AM, PA) have a single seller each.
- 611 distinct city names. `sao paulo` alone has 694 sellers, then `curitiba` (127), `rio de janeiro` (96), `belo horizonte` (68) and `ribeirao preto` (52).
- 342 cities (56.0%) have a single seller. The long tail is thin.
- Customers are spread over 27 states with 42% in SP, so many orders cross state lines. Distances are computed after the joins.

#### City text quality
**Question:** is the city text clean?

In [5]:
def strip_accents(s):
    return "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch)).lower().strip()

city = sellers["seller_city"]
print("city values with upper case      :", (city != city.str.lower()).sum())
print("city values with extra spaces    :", (city != city.str.strip()).sum())
print("city values with non-ASCII chars :", (~city.map(str.isascii)).sum())
print("distinct names raw / without accents:", city.nunique(), "/", city.map(strip_accents).nunique())

only_digits = city[city.str.fullmatch(r"\d+")]
print("\ncity values that are only digits :", len(only_digits), "->", only_digits.tolist())

has_symbol = city[city.str.contains(r"[@/\\\-(),.]")]
print("city values with symbols         :", len(has_symbol))
print(has_symbol.drop_duplicates().head(10).tolist())

# How is Sao Paulo written?
sp_like = city[city.map(strip_accents).str.contains("sao paulo|^sp$|sbc")]
sp_like.value_counts().head(8).rename("n_sellers").to_frame()

city values with upper case      : 0
city values with extra spaces    : 0
city values with non-ASCII chars : 3
distinct names raw / without accents: 611 / 609

city values that are only digits : 1 -> ['04482255']
city values with symbols         : 25
['lages - sc', 'auriflama/sp', 'sao paulo / sao paulo', 'novo hamburgo, rio grande do sul, brasil', 'cariacica / es', 'sao paulo - sp', 'sbc/sp', "arraial d'ajuda (porto seguro)", 'santo andre/sao paulo', 'sp / sp']


,n_sellers
seller_city,
sao paulo,694
sp,4
sao paulo - sp,3
sao paulo / sao paulo,1
são paulo,1
sbc/sp,1
santo andre/sao paulo,1
maua/sao paulo,1


- The basics are clean: all lower case and no extra spaces. 3 values have accents, and removing accents merges only 2 names (611 to 609).
- 1 city value is only digits (`04482255`), which looks like a zip code typed in the city field.
- 25 city values contain symbols and look like free text: `lages - sc`, `sao paulo / sao paulo`, `sp / sp`, `sbc/sp`, `novo hamburgo, rio grande do sul, brasil`. They carry the state or a second city inside the name.
- São Paulo is written in several ways: `sao paulo` (694), `sp` (4), `sao paulo - sp` (3), and single entries such as `são paulo` and `sao paulo / sao paulo`.
- **Decision:** do not rely on the city text. The zip prefix is the location key. Hand-cleaning 25 values adds guesses, and the prefix already identifies the place.

#### Consistency of zip, city and state
**Question:** do zip prefix, city and state agree with each other?

In [6]:
by_zip = sellers.groupby("seller_zip_code_prefix").agg(n_cities=("seller_city", "nunique"),
                                                      n_states=("seller_state", "nunique"),
                                                      n_sellers=("seller_id", "size"))
print("zip prefixes in the table                 :", len(by_zip))
print("zip prefixes with more than one state     :", (by_zip["n_states"] > 1).sum(),
      "| sellers in them:", by_zip.loc[by_zip["n_states"] > 1, "n_sellers"].sum())
print("zip prefixes with more than one city name :", (by_zip["n_cities"] > 1).sum())
print("city names used in more than one state    :", (sellers.groupby("seller_city")["seller_state"].nunique() > 1).sum())

print("\nexamples: same zip prefix, different state")
conflict_zip = by_zip.index[by_zip["n_states"] > 1]
display(sellers[sellers["seller_zip_code_prefix"].isin(conflict_zip)]
        .sort_values(["seller_zip_code_prefix", "seller_state"])[["seller_zip_code_prefix", "seller_city", "seller_state"]].head(8))

print("examples: same zip prefix, different city text")
multi_city = sellers.groupby("seller_zip_code_prefix")["seller_city"].agg(lambda s: tuple(sorted(s.unique())))
multi_city[by_zip["n_cities"] > 1].head(5).to_frame("city names")

zip prefixes in the table                 : 2246
zip prefixes with more than one state     : 17 | sellers in them: 45
zip prefixes with more than one city name : 34
city names used in more than one state    : 23

examples: same zip prefix, different state


,seller_zip_code_prefix,seller_city,seller_state
1591,21210,rio de janeiro,RJ
114,21210,rio de janeiro,RN
299,22783,rio de janeiro,RJ
364,22783,rio de janeiro,SP
1693,31160,belo horizonte,MG
3036,31160,belo horizonte,SP
2509,36010,juiz de fora,MG
3012,36010,juiz de fora,MG


examples: same zip prefix, different city text


,city names
seller_zip_code_prefix,
1207,"(sao paulo, sao paulo sp)"
5141,"(pirituba, sp)"
7077,"(garulhos, guarulhos)"
8710,"(mogi das cruses, mogi das cruzes)"
9190,"(sando andre, santo andre)"


- 17 zip prefixes (45 sellers) are filed under more than one state. A zip prefix belongs to one state, so at least one state is wrong in each case.
- The examples show it: prefix 21210 is `rio de janeiro` with state RJ for one seller and RN for another, prefix 22783 is `rio de janeiro` under RJ and SP, and prefix 31160 is `belo horizonte` under MG and SP. The city and zip agree and the state looks mistyped.
- 23 city names are used in more than one state (for example `belo horizonte` in MG and SP, as in the table above). The same state typos explain it.
- 34 zip prefixes have more than one city text. They are spelling variants and typos (`garulhos` / `guarulhos`, `mogi das cruses`, `sando andre`) or a place and its district (`sp` / `pirituba`).
- **Decision:** treat `seller_state` as less reliable than the zip prefix. Keep all sellers, and cross-check the state against the geolocation table after the joins.

#### Summary
**The story in short.** The sellers table is a small clean key table (3,095 sellers, no missing values, no duplicates), but its location text is messy. Sellers are far more concentrated than customers (59.7% in SP), and many sellers share one zip prefix. The city text has free-text entries, and the state disagrees with the zip prefix for 45 sellers. The zip prefix is the most reliable location field, apart from the lost leading zero.

**Data quality issues**

| # | Issue | Records | What it is | Action | Cost of the alternative |
|---|---|---|---|---|---|
| 1 | Zip prefix lost its leading zero | 1,027 (33.2%), all SP | Stored as a number | Keep numeric for the geo join, add `zip5` text for display | A text zip joined to a numeric zip silently matches nothing |
| 2 | State disagrees with zip prefix | 17 prefixes, 45 sellers | State mistyped | Keep, cross-check the state against the geolocation table after the joins | Trusting the state gives wrong region features |
| 3 | City text with symbols or extra place names | 25 rows | Free-text entries | Do not use city text. Use the zip prefix. | Hand-cleaning adds guesses |
| 4 | City value that is a zip code | 1 (`04482255`) | Wrong field filled | Keep, ignore the city text | Dropping the seller loses its orders later |
| 5 | `sp` and other variants for São Paulo | 4 `sp`, 3 `sao paulo - sp`, and single entries | Inconsistent spelling | Use the zip prefix | Grouping by city text splits São Paulo |
| 6 | One zip prefix, several city texts | 34 prefixes | Spelling variants and districts | Use zip prefix as the key | Merging names risks wrong merges |
| 7 | City name used in several states | 23 names | State typos | Never use city alone as a key | Wrong merges between states |
| 8 | Thin location tail | 342 cities (56.0%) with one seller | Many small cities | Use state or distance bands instead of city | Rates per city are unstable |

**Checks planned after the joins** (not done here on purpose)
- Every seller zip prefix against the geolocation table, and the seller state against the state of that prefix.
- Every seller in the items table, and sellers without any sold item.
- Seller history (delay rate, bad-review rate) built from past orders only.
- Distance between seller and customer prefixes.

**What can be used at order approval (for Part 2)**

| Field | Known at approval? | Note |
|---|---|---|
| `seller_zip_code_prefix` (with `zip5`) | Yes | Basis for the distance feature. Most reliable location field. |
| `seller_state` | Yes | Less reliable than the prefix (45 conflicting sellers) |
| `seller_city` | Yes | Dirty text, display only |
| `seller_id` | Yes | Key for seller history. History must use only orders closed before the order being scored. |